# BioImage Archive submission from an OMERO annotation table

Select an annotation table on OMERO, pull its metadata and data down, and package a
**BioImage Archive (BIA)** submission bundle: MIFA metadata (`Study`/`Annotations`/`Version`
YAML), the BIA file lists (`file_list_images.tsv`, `file_list_annotations.tsv`), and the
image/mask files, all in one self-contained folder.

Same starting point as the training notebooks — pick a table — but instead of training a
model, this prepares an archive submission. The only thing written to OMERO is the
description you complete in step 4 (`ts.describe`), which is saved with the set's config.

For the offline variant (data already on this machine from an annotation run), see
`prepare_bioimage_archive.ipynb`.

> Requires the optional `bia-mifa-models` package (in the `dev`/`mifa` pixi environments).

## 1. Setup

In [ ]:
import datetime
from pathlib import Path

import pandas as pd

from omero_annotate_ai import (
    TrainingSet,
    create_omero_connection_widget,
    create_training_data_widget,
    prepare_bia_data_from_table,
)
from omero_annotate_ai.schema_export import save_bia_package

## 2. OMERO connection

In [ ]:
conn_widget = create_omero_connection_widget()
conn_widget.display()

In [ ]:
conn = conn_widget.get_connection()

if conn is None:
    raise ConnectionError("No OMERO connection established.")

print(f"Connected to OMERO as: {conn.getUser().getName()}")

## 3. Select the training set

Pick the container and the training set to publish. Its config on OMERO already holds
the description (study, authors, licence, methodology) you filled in earlier, if any.

In [ ]:
table_widget = create_training_data_widget(connection=conn, show_export=False)
table_widget.display()

In [ ]:
table_id = table_widget.get_selected_table_id()
if not table_id:
    raise ValueError(
        "Select a training set on OMERO above (for a local set, use "
        "prepare_bioimage_archive.ipynb)."
    )
ts = TrainingSet.open(table_id, conn)
table_name = ts.name
print(f"Selected training set: {table_name} (table {table_id})")

## 4. Review and complete the description

MIFA requires a study title, description and funding statement, keywords, at least one
link (or a source dataset URL) and a version. The annotation widget does not ask for
these, so a set made there has none yet. Fill in below what is missing; a field you leave
empty keeps what the set already has. `ts.describe` saves them with the set, locally and
on OMERO. Nothing is invented for you: if a required field is still missing,
`save_bia_package` stops with `MifaNotReady` and names it.

In [ ]:
# --- fill these in (empty = keep what the set already has) --------------------
title = ""
description = ""
funding_statement = ""
organism = ""  # optional
version = ""  # e.g. "1"
license = ""  # "CC0-1.0" or "CC-BY-4.0": MIFA allows only these
keywords: list[str] = []
# e.g. [{"first_name": "...", "last_name": "...", "affiliations": [{"name": "..."}]}]
authors: list[dict] = []
# e.g. [{"url": "https://...", "description": "Analysis code"}]
links: list[dict] = []
# The S-BIAD id once the BIA has issued one; until then the set's name is used in the
# MIFA filenames.
accession = ""
# ----------------------------------------------------------------------------

study = {
    **(ts.config.study.model_dump(exclude_none=True) if ts.config.study else {}),
    **{
        key: value
        for key, value in {
            "title": title,
            "description": description,
            "funding_statement": funding_statement,
            "organism": organism,
        }.items()
        if value
    },
}
filled = {
    "version": version,
    "license": license,
    "keywords": keywords,
    "authors": authors,
    "links": links,
    "study": study,
}
ts.describe(**{key: value for key, value in filled.items() if value})
config = ts.config

print(f"Title:     {config.study.title if config.study else ''}")
print(f"License:   {config.license}")
print(
    f"Authors:   {', '.join(f'{a.first_name} {a.last_name}' for a in config.authors)}"
)
print(f"Funding:   {config.study.funding_statement if config.study else ''}")
print(f"Keywords:  {', '.join(config.keywords)}")
print(f"Version:   {config.version}")
print(f"Accession: {accession or '(not issued yet)'}")

## 5. Download the images and masks from OMERO

Pulls each annotated plane/patch and its mask into `annotation_input/` and
`annotation_output/`, at their **native bit depth**. Rows that were never processed, or that
carry no mask, are skipped and reported. The returned `rows` are exactly the exported records.

In [ ]:
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
run_dir = Path.home() / "omero-annotate-ai" / "bia-export" / f"{table_name}_{timestamp}"
staging_dir = run_dir / "omero_data"  # scratch: what we pull off OMERO
bundle_dir = run_dir / "submission"  # what you send to the BioImage Archive

stats = prepare_bia_data_from_table(
    conn=conn, table_id=table_id, output_dir=staging_dir
)
rows = stats["rows"]

print(f"Downloaded {len(rows)} units to: {staging_dir}")
for key in ("n_images", "n_masks", "skipped_unprocessed", "skipped_no_mask"):
    print(f"  {key}: {stats[key]}")

## 6. Build the BIA submission bundle

The bundle uses the folder names a reader of the archive sees: `images/` for the raw planes
and `annotations/` for the masks (the file lists carry paths relative to the submission root).

`move_data=True` **moves** the downloaded files into the bundle instead of copying them, so a
large dataset is not held on disk twice. That empties `staging_dir`, which is fine here because
it is a scratch copy pulled from OMERO. Never pass it with a training set's own folder.

In [ ]:
result = save_bia_package(
    config, rows, staging_dir, bundle_dir, accession=accession or None, move_data=True
)
print("Bundle written to:", result["dir"])
print(
    f"  images: {result['n_images']} | annotations: {result['n_annotations']} "
    f"| files transferred: {result['copied']} | missing: {result['missing']}"
)
if result["missing"]:
    print("\nWARNING: the file lists reference files that are not in the bundle.")

print("\nBundle contents:")
for path in sorted(result["dir"].rglob("*")):
    if path.is_file():
        print("  ", path.relative_to(result["dir"]))

## 7. Inspect the file lists and MIFA metadata

Check these before submitting. Every row of `file_list_annotations.tsv` must point at a real
mask file, and its `source_image` at the raw image that mask belongs to.

In [ ]:
print("=== file_list_annotations.tsv ===")
print(pd.read_csv(result["file_list_annotations"], sep="\t").to_string(index=False))
print("\n=== file_list_images.tsv ===")
print(pd.read_csv(result["file_list_images"], sep="\t").to_string(index=False))
print(f"\n=== metadata/{result['metadata']['annotations'].name} ===")
print(result["metadata"]["annotations"].read_text())

## 8. Upload to the BioImage Archive

Submit the `submission/` folder (file lists + data + `metadata/`) to the BioImage Archive via
their FTP/Aspera/Globus transfer, following
[the BIA submission guide](https://www.ebi.ac.uk/bioimage-archive/help-file-list/).

Once published you receive an accession (`S-BIAD###`) and a DOI. Add them to the set with
`ts.describe(links=[*ts.config.links, {"url": "<BIA study URL>", "description": "BioImage Archive"}])`,
so a later BioImage Model Zoo export can cite this dataset as the model's `training_data`.

## 9. Cleanup

In [ ]:
if conn is not None:
    conn.close()
    print("OMERO connection closed.")